In [ ]:

import os
import gc
import cv2
import torch
import random
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, random_split
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import xml.etree.ElementTree as ET


def extract_text_crops(image_path, xml_path):
    """
    Returns list of (image_path, xmin, ymin, xmax, ymax, label) tuples.
    Images are NOT loaded into memory here — only paths and coordinates.
    """
    if not Path(image_path).exists():
        return []

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return []

    samples = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        if name_el is None or name_el.text != 'text':
            continue

        text_el = obj.find('text')
        if text_el is None or not text_el.text:
            continue

        ground_truth = text_el.text.strip()
        if not ground_truth:
            continue

        bndbox = obj.find('bndbox')
        try:
            xmin = int(bndbox.find('xmin').text)
            ymin = int(bndbox.find('ymin').text)
            xmax = int(bndbox.find('xmax').text)
            ymax = int(bndbox.find('ymax').text)
        except (ValueError, AttributeError):
            continue

        samples.append((str(image_path), xmin, ymin, xmax, ymax, ground_truth))

    return samples


class SchematicTextDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, xmin, ymin, xmax, ymax, label = self.samples[idx]

        img = cv2.imread(image_path)
        if img is None:
            img = np.zeros((32, 32, 3), dtype=np.uint8)

        pad = 5
        h, w = img.shape[:2]
        xmin_p = max(0, xmin - pad)
        ymin_p = max(0, ymin - pad)
        xmax_p = min(w, xmax + pad)
        ymax_p = min(h, ymax + pad)

        crop = img[ymin_p:ymax_p, xmin_p:xmax_p]
        if crop.size == 0:
            crop = np.zeros((32, 32, 3), dtype=np.uint8)

        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb).convert("RGB")

        pixel_values = self.processor(
            pil_img, return_tensors="pt"
        ).pixel_values.squeeze()

        encoding = self.processor.tokenizer(
            label,
            padding="max_length",
            max_length=32,
            truncation=True,
        )
        labels = encoding.input_ids

        pad_id = self.processor.tokenizer.pad_token_id
        labels = [l if l != pad_id else -100 for l in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels, dtype=torch.long),
        }



print("Scanning dataset for text annotations...")

DATASET_ROOT = Path('/kaggle/input/datasets/johannesbayer/cghd1152')

all_samples = []
xml_files = list(DATASET_ROOT.rglob('*.xml'))
print(f"Found {len(xml_files)} XML files")

for xml_file in xml_files:
    image_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.jpg')
    if not image_file.exists():
        image_file = xml_file.parent.parent / 'images' / (xml_file.stem + '.png')
    if not image_file.exists():
        continue
    crops = extract_text_crops(image_file, xml_file)
    all_samples.extend(crops)

print(f"Total text crop samples: {len(all_samples)}")

random.shuffle(all_samples)


for image_path, xmin, ymin, xmax, ymax, label in all_samples[:10]:
    print(f"  Label: '{label}'")

print("\nLoading TrOCR model...")
MODEL_CHECKPOINT = 'microsoft/trocr-small-handwritten'

processor = TrOCRProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_CHECKPOINT)

# Decoder config — only non-generation parameters go in model.config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id

# Generation parameters go in generation_config
model.generation_config.max_length = 32
model.generation_config.early_stopping = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty = 2.0
model.generation_config.num_beams = 4


train_size = int(0.8 * len(all_samples))
val_size = len(all_samples) - train_size
train_samples, val_samples = random_split(all_samples, [train_size, val_size])

train_dataset = SchematicTextDataset(list(train_samples), processor)
val_dataset = SchematicTextDataset(list(val_samples), processor)

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")

OUTPUT_DIR = '/kaggle/working/trocr-schematic'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
)


gc.collect()
torch.cuda.empty_cache()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\nStarting training...")
trainer.train()


FINAL_MODEL_DIR = '/kaggle/working/trocr-schematic-final'
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

print(f"Model saved to {FINAL_MODEL_DIR}")
